# Zero-Shot Model Comparison – Chronos & Kronos

Runs **Chronos-bolt-base** and **Kronos-base** in zero-shot mode on energy-sector financial data (test split: 2021+).
Both models are evaluated across 5 seeds; an additional Kronos run uses the optimal inference parameters
identified in the data-parameter sensitivity analysis for a fair comparison against the fine-tuned variant.

| Setting | Value | Source |
|---|---|---|
| Seeds | 13, 42, 123, 456, 789 | Multi-seed robustness |
| Config | `config/energy_assets_filtered.yaml` | 104-Asset-Universum |
| Context | 80 | HANDOFF |
| Forecast | 12 | HANDOFF |

**Workflow:**
1. Setup environment
2. Run Chronos zero-shot (5 seeds)
3. Run Kronos zero-shot (5 seeds)
4. Aggregate per-seed metrics
5. Kronos base run with optimal params (context=120, forecast=6) for fine-tuning comparison

## Setup: Clone Repository

In [1]:
!rm -rf /content/BA
print("Cleaned up.")

Cleaned up.


In [2]:
import os
from pathlib import Path

repo_url = "https://github.com/bp571/BA"

%cd /content
!git clone {repo_url}
%cd /content/BA

TIINGO_API_KEY = "312c6dab6a1fe6258bbc6652bcdec49a14ee08ad"
os.environ["TIINGO_API_KEY"] = TIINGO_API_KEY
Path(".env").write_text(f"TIINGO_API_KEY={TIINGO_API_KEY}\n")
print("✅ API key configured")

# Kronos source repo (gitignored im Hauptrepo)
!git clone https://github.com/shiyu-coder/Kronos.git 02_finetuning/models/Kronos
print("✅ Kronos cloned")

/content
Cloning into 'BA'...
remote: Enumerating objects: 766, done.
remote: Counting objects: 100% (237/237), done.
remote: Compressing objects: 100% (196/196), done.
remote: Total 766 (delta 60), reused 209 (delta 40), pack-reused 529 (from 1)
Receiving objects: 100% (766/766), 24.54 MiB | 28.55 MiB/s, done.
Resolving deltas: 100% (296/296), done.
/content/BA
✅ API key configured
Cloning into '02_finetuning/models/Kronos'...
remote: Enumerating objects: 371, done.
remote: Counting objects: 100% (131/131), done.
remote: Compressing objects: 100% (55/55), done.
remote: Total 371 (delta 86), reused 76 (delta 76), pack-reused 240 (from 1)
Receiving objects: 100% (371/371), 9.30 MiB | 37.64 MiB/s, done.
Resolving deltas: 100% (183/183), done.
✅ Kronos cloned


## Install Dependencies

In [3]:
!pip install -q torch transformers peft gluonts pandas numpy tqdm PyYAML \
    chronos-forecasting einops huggingface_hub yfinance scipy python-dotenv tiingo

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.0/44.0 kB 4.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.5/1.5 MB 52.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 72.7/72.7 kB 8.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.0/12.0 MB 119.4 MB/s eta 0:00:0000:010:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 48.8 MB/s eta 0:00:00


In [4]:
import torch
print(f"PyTorch: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")

PyTorch: 2.10.0+cu128
CUDA available: True
GPU: NVIDIA A100-SXM4-80GB


## Configuration

Zero-Shot Parameter (HANDOFF: context=80, forecast=12, stride=12) über 5 Seeds auf dem 104-Asset-Universum.

In [5]:
import sys
from pathlib import Path

project_root = Path.cwd()
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

# Re-Run auf 104-Asset-Universum (energy_assets_filtered.yaml)
SEEDS    = [13, 42, 123, 456, 789]
CONFIG   = "config/energy_assets_filtered.yaml"

# Zero-Shot Parameter (HANDOFF: c=80, f=12, s=12)
CONTEXT  = 80
FORECAST = 12

print(f"Seeds:    {SEEDS}")
print(f"Config:   {CONFIG}")
print(f"Context:  {CONTEXT}, Forecast: {FORECAST}")

Seeds:    [13, 42, 123, 456, 789]
Config:   config/energy_assets_filtered.yaml
Context:  80, Forecast: 12


## Chronos Zero-Shot

Runs Chronos-bolt-base across all 5 seeds. Results are written to `01_model_comparison/results/chronos/seed_<seed>/`.

In [6]:
import subprocess

# Chronos-2 ist deterministisch (Quantil-Output, kein Sampling) -> 1 Seed reicht
CHRONOS_SEED = SEEDS[0]
print(f"Starte Chronos-2 Single-Seed Run (seed={CHRONOS_SEED})...")
res = subprocess.run(
    ["python", "01_model_comparison/zeroshot/main_chronos.py",
     "--seed", str(CHRONOS_SEED), "--config", CONFIG],
    capture_output=True, text=True
)
print(f"\n{'='*60}\nChronos | Seed {CHRONOS_SEED} | rc={res.returncode}\n{'='*60}")
print(res.stdout[-3000:])
if res.returncode != 0:
    print("STDERR:", res.stderr[-2000:])

Starte Chronos-2 Single-Seed Run (seed=13)...

Chronos | Seed 13 | rc=0
: 4 gaps > 5 days detected in time series
⚠️  0857.HK: 16 outlier returns (>3 std) detected
⚠️  EC: 19 outlier returns (>3 std) detected
⚠️  SSL: 16 outlier returns (>3 std) detected
⚠️  PTT.BK: 5 gaps > 5 days detected in time series
⚠️  PTT.BK: 14 outlier returns (>3 std) detected
⚠️  MOL.BD: 3 gaps > 5 days detected in time series
⚠️  MOL.BD: 24 outlier returns (>3 std) detected
⚠️  PKN.WA: 1 gaps > 5 days detected in time series
⚠️  PKN.WA: 16 outlier returns (>3 std) detected
⚠️  6505.TW: 8 gaps > 5 days detected in time series
⚠️  6505.TW: 24 outlier returns (>3 std) detected
⚠️  SM: 18 outlier returns (>3 std) detected
⚠️  REI: 17 outlier returns (>3 std) detected
⚠️  FSLR: 20 outlier returns (>3 std) detected
⚠️  NEE: 19 outlier returns (>3 std) detected
⚠️  PLUG: 18 outlier returns (>3 std) detected
⚠️  CCJ: 17 outlier returns (>3 std) detected
⚠️  FCEL: 17 outlier returns (>3 std) detected
⚠️  RWEOY: 17 o

## Kronos Zero-Shot

Runs Kronos-base across all 5 seeds. Results are written to `01_model_comparison/results/kronos/seed_<seed>/`.

In [ ]:
import subprocess
from concurrent.futures import ThreadPoolExecutor, as_completed

def run_kronos(seed):
    res = subprocess.run(
        ["python", "01_model_comparison/zeroshot/main_kronos.py",
         "--seed", str(seed), "--config", CONFIG,
         "--context", str(CONTEXT), "--forecast", str(FORECAST)],
        capture_output=True, text=True
    )
    return seed, res

print(f"Starte {len(SEEDS)} Kronos-Runs parallel...")
with ThreadPoolExecutor(max_workers=len(SEEDS)) as ex:
    futures = [ex.submit(run_kronos, s) for s in SEEDS]
    for fut in as_completed(futures):
        seed, res = fut.result()
        print(f"\n{'='*60}\nKronos | Seed {seed} | rc={res.returncode}\n{'='*60}")
        print(res.stdout[-2000:])
        if res.returncode != 0:
            print("STDERR:", res.stderr[-2000:])

## Summary

In [ ]:
import json
import numpy as np
import pandas as pd
from pathlib import Path

def aggregate(model: str, seed: int) -> dict | None:
    p = Path(f"01_model_comparison/results/{model}/seed_{seed}/final_energy_study.json")
    if not p.exists():
        return None
    with open(p) as f:
        data = json.load(f)
    summary = data["summary"]
    maes    = [v["MAE_indicative"]         for v in summary.values() if "MAE_indicative"         in v]
    ics     = [v["IC_TimeSeries_Mean"]     for v in summary.values() if "IC_TimeSeries_Mean"     in v]
    rankics = [v["RankIC_TimeSeries_Mean"] for v in summary.values() if "RankIC_TimeSeries_Mean" in v]
    return {
        "model":    model,
        "seed":     seed,
        "n_assets": data["n_assets_processed"],
        "time_s":   data["processing_time_seconds"],
        "MAE":      np.mean(maes)    if maes    else np.nan,
        "IC":       np.mean(ics)     if ics     else np.nan,
        "RankIC":   np.mean(rankics) if rankics else np.nan,
    }

rows = []
for model in ["chronos", "kronos"]:
    for seed in SEEDS:
        row = aggregate(model, seed)
        if row:
            rows.append(row)

df = pd.DataFrame(rows)
print("\nPer-Seed Ergebnisse")
print("="*70)
print(df.round(4).to_string(index=False))

print("\nAggregat (Mittelwert ± Std über Seeds)")
print("="*70)
agg = df.groupby("model").agg(
    n_assets=("n_assets", "mean"),
    MAE_mean=("MAE", "mean"), MAE_std=("MAE", "std"),
    IC_mean=("IC", "mean"),   IC_std=("IC", "std"),
    RankIC_mean=("RankIC", "mean"), RankIC_std=("RankIC", "std"),
    total_time_s=("time_s", "sum"),
).round(4)
print(agg.to_string())

## Download Results

In [ ]:
from google.colab import files
import zipfile
from pathlib import Path

zip_name = "zeroshot_results.zip"

with zipfile.ZipFile(zip_name, 'w') as zf:
    for model in ["chronos", "kronos"]:
        results_root = Path(f"01_model_comparison/results/{model}")
        if not results_root.exists():
            continue
        for f in results_root.rglob("*.json"):
            zf.write(f, arcname=str(f.relative_to("01_model_comparison/results")))

files.download(zip_name)
print(f"Downloaded: {zip_name}")